# Photon counting with a Hidden Markov Model

Replication of the hidden Markov model (HMM) analysis of

> A. V. Dixit, S. Chakram, K. He, A. Agrawal, R. K. Naik, D. I. Schuster, A. Chou,
> *Searching for Dark Matter with a Superconducting Qubit*,
> **Phys. Rev. Lett. 126, 141302 (2021)** — [arXiv:2008.12231](https://arxiv.org/abs/2008.12231)

The experiment counts single microwave photons in a storage cavity by repeatedly
asking a transmon qubit about the cavity's photon number, without absorbing the
photon. Each individual question is unreliable (the readout is ~95% faithful, the
qubit decays, it heats), but the questions are cheap and can be repeated. The HMM
is what turns many unreliable answers into one reliable one, and it is what buys
the paper its 15.7 dB of noise reduction below the standard quantum limit.

**What this notebook does**

1. Builds the transition matrix $T$ (Eqn. 2) and the emission matrix $E$ (Eqn. 3).
2. Evolves the model with no measurements at all, to see the bare dynamics.
3. Runs the **forward filter** — "is a photon here *now*?"
4. Runs the **backward algorithm** (Eqn. 4) — "was a photon there *at the start*?",
   which is the quantity Fig. 2(c) plots.
5. Turns that into a likelihood ratio and a detection threshold.

---

## Notation

The hidden state is the *joint* cavity-qubit state,

$$X \in \mathbf{X} := \{\,|0g\rangle,\; |0e\rangle,\; |1g\rangle,\; |1e\rangle\,\}$$

where the first index is the photon number in the storage cavity ($0$ or $1$, valid
while $\bar n \ll 1$) and the second is the transmon state. Throughout the code these
four states are always kept **in this order**, and every vector of length 4 is indexed
by it.

What we actually observe is the readout,

$$Y \in \mathbf{Y} := \{\mathcal{G}, \mathcal{E}\}$$

encoded in the code as `R = 0` for $\mathcal{G}$ and `R = 1` for $\mathcal{E}$.

A run of the experiment produces a **record** of $N+1$ readouts
$R = \{R_0, R_1, \ldots, R_N\}$ generated by a hidden trajectory
$S = \{X_0, X_1, \ldots, X_N\}$ that we never see. Everything below is about
recovering statements about $S$ from $R$.

The cavity state is *not* destroyed by the measurement: the qubit-cavity interaction
$2\chi a^\dagger a\, \sigma_z / 2$ commutes with the cavity number operator, so the
measurement is QND and the same photon can be interrogated over and over. That is the
whole reason repetition is available to us here.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## 1. The model

### 1.1 Error channels

One measurement cycle lasts $t_m$ (the time between parity measurements) and contains
a parity mapping of length $t_p$. Over one cycle:

**Qubit.** Relaxation $|e\rangle \to |g\rangle$ happens with probability
$P_{eg}^{\downarrow} = 1 - e^{-t_m/T_1^q}$, and the spurious excited-state population
$\bar n_q$ drives the reverse, $P_{ge}^{\uparrow} = \bar n_q\,(1 - e^{-t_m/T_1^q})$.
Dephasing during the parity measurement adds
$P_\phi = 1 - e^{-t_p/T_2^q}$ to both, since a dephased qubit gives a random answer:

$$P_{eg} = P_{eg}^{\downarrow} + P_\phi, \qquad
  P_{ge} = P_{ge}^{\uparrow} + P_\phi, \qquad
  P_{gg} = 1 - P_{ge}, \qquad
  P_{ee} = 1 - P_{eg}$$

**Cavity.** Identically, with the storage lifetime $T_1^s$ and residual occupation $\bar n_c$:

$$P_{10} = 1 - e^{-t_m/T_1^s}, \qquad
  P_{01} = \bar n_c\,(1 - e^{-t_m/T_1^s}), \qquad
  P_{00} = 1 - P_{01}, \qquad
  P_{11} = 1 - P_{10}$$

### 1.2 Transition matrix — Eqn. (2)

$$
T=\begin{array}{ccccc}
|0g\rangle & |0e\rangle & |1g\rangle & |1e\rangle \\
P_{00}P_{gg} & P_{00}P_{ge} & P_{01}P_{ge} & P_{01}P_{gg} & \;|0g\rangle\\
P_{00}P_{eg} & P_{00}P_{ee} & P_{01}P_{ee} & P_{01}P_{eg} & \;|0e\rangle\\
P_{10}P_{gg} & P_{10}P_{ge} & P_{11}P_{ge} & P_{11}P_{gg} & \;|1g\rangle\\
P_{10}P_{eg} & P_{10}P_{ee} & P_{11}P_{ee} & P_{11}P_{eg} & \;|1e\rangle\\
\end{array}
$$

$T_{ij} = P(X_{k+1} = j \mid X_k = i)$, so **rows sum to 1**. Read it as
(cavity move) $\times$ (qubit move), *except* that the qubit labels are swapped in
the two $|1\cdot\rangle$ columns — see §1.4, that swap is the physics.

### 1.3 Emission matrix — Eqn. (3)

$$
E = \frac{1}{2}\begin{array}{ccc}
\mathcal{G} & \mathcal{E} \\
F_{g\mathcal{G}} & F_{g\mathcal{E}} & \;|0g\rangle\\
F_{e\mathcal{G}} & F_{e\mathcal{E}} & \;|0e\rangle\\
F_{g\mathcal{G}} & F_{g\mathcal{E}} & \;|1g\rangle\\
F_{e\mathcal{G}} & F_{e\mathcal{E}} & \;|1e\rangle\\
\end{array}
$$

$E_{iR} = P(Y = R \mid X = i)$ — the readout fidelities, $F_{g\mathcal{G}} = 95.8\%$ and
$F_{e\mathcal{E}} = 95.3\%$. The rows for $|1g\rangle, |1e\rangle$ are **identical** to
those for $|0g\rangle, |0e\rangle$: the readout resonator only ever sees the qubit, never
the storage cavity. All the cavity information reaches us through the qubit, and only
through the parity map.

`D(R, args)` returns column $R$ of $E$ laid out as a diagonal matrix, so that
`D(R) @ p` multiplies each state's probability by its likelihood $P(Y=R \mid X)$.

The $\tfrac12$ is in the paper's Eqn. (3). It cancels in every normalisation we do,
so it never changes a result — but it does mean the literal value of `np.sum(x)`
inside `HMM_step` is $\tfrac12 P(Y)$ rather than $P(Y)$.

In [ ]:
# ---------------------------------------------------------------------------
# Error probabilities over one measurement cycle (see markdown above).
# All of these are per-cycle probabilities, not rates.
# ---------------------------------------------------------------------------

# Qubit relaxation, heating and dephasing
Peg_   = lambda t_m, args: (1 - np.exp(-t_m/args["T1_q"]))                   # |e> -> |g>, T1 decay
Pge_   = lambda t_m, args: (1 - np.exp(-t_m/args["T1_q"])) * args["n_q"]     # |g> -> |e>, thermal
P_phi_ = lambda t_p, args: 1 - np.exp(-t_p/args["T2_q"])                     # dephasing during t_p
Peg = lambda t_m, t_p, args: Peg_(t_m, args) + P_phi_(t_p, args)   # a dephased qubit answers at random,
Pge = lambda t_m, t_p, args: Pge_(t_m, args) + P_phi_(t_p, args)   # so P_phi adds to both flip directions
Pgg = lambda t_m, t_p, args: 1 - Pge(t_m, t_p, args)
Pee = lambda t_m, t_p, args: 1 - Peg(t_m, t_p, args)

# Storage cavity: same structure, with T1_c and the residual occupation n_c
P10 = lambda t_m, args: 1 - np.exp(-t_m/args["T1_c"])                        # photon lost
P01 = lambda t_m, args: args["n_c"] * (1 - np.exp(-t_m/args["T1_c"]))        # photon gained
P00 = lambda t_m, args: 1 - P01(t_m, args)
P11 = lambda t_m, args: 1 - P10(t_m, args)


def T(t_m, t_p, args):
    """Transition matrix, Eqn. (2). T[i, j] = P(X_{k+1} = j | X_k = i), rows sum to 1.

    The two columns for 1 photon in the cavity have the qubit labels SWAPPED.
    That swap is the parity map: in the experiment the pi/2 -- wait t_p -- -pi/2
    sequence is a pi rotation of the QUBIT conditioned on the photon. Starting
    from |0g>, nothing happens and the qubit comes back to |g>. Starting from
    |1g>, the photon detunes the qubit by 2*chi and the wait accumulates exactly
    pi of phase, so the qubit ends in |e>. This is how a measurement of the qubit
    becomes a measurement of the cavity.

    Concretely, in the |1.> columns the "no error" outcome is the FLIPPED qubit:
    |1g> -> |1e> carries Pgg (dominant, ~0.93) while |1g> -> |1g> carries Pge,
    because staying put requires a qubit error to undo the flip.

    Drop the swap and T factorises as kron(T_cavity, T_qubit): the qubit never
    correlates with the photon and the record carries no information about n.
    """
    gg = Pgg(t_m, t_p, args); ge = Pge(t_m, t_p, args);
    eg = Peg(t_m, t_p, args); ee = Pee(t_m, t_p, args);
    c00 = P00(t_m, args); c01 = P01(t_m, args);
    c10 = P10(t_m, args); c11 = P11(t_m, args);

    #                   |0g>    |0e>    |1g>    |1e>
    return np.array([   [gg*c00, ge*c00, ge*c01, gg*c01], #|0g>
                        [eg*c00, ee*c00, ee*c01, eg*c01], #|0e>
                        [gg*c10, ge*c10, ge*c11, gg*c11], #|1g>
                        [eg*c10, ee*c10, ee*c11, eg*c11]]) #|1e>


def E(args):
    """Emission matrix, Eqn. (3). E[i, R] = P(Y = R | X = i).

    The |1g>, |1e> rows are IDENTICAL to the |0g>, |0e> rows: the readout
    resonator is coupled to the qubit only and never sees the storage cavity.
    Every bit of cavity information reaches us through the parity map in T.

    The 1/2 prefactor is in the paper's Eqn. (3). It cancels in every
    normalisation, but it does mean np.sum(...) inside HMM_step evaluates to
    (1/2) P(Y) rather than P(Y).
    """
    #                     G            E
    return 0.5 * np.array( [[args["Fgg"], args["Fge"]], #|0g>
                            [args["Feg"], args["Fee"]], #|0e>
                            [args["Fgg"], args["Fge"]], #|1g>
                            [args["Feg"], args["Fee"]]]) #|1e>


def D(R, args):
    """Column R of E as a diagonal matrix, so that D(R) @ p applies P(Y=R | X)
    entry by entry. R = 0 is a ground-state readout, R = 1 an excited one."""
    E_matrix = E(args)

    if R == 0:
        return np.diag(E_matrix[:, 0])
    elif R == 1:
        return np.diag(E_matrix[:, 1])

### 1.4 Why the two $|1\rangle$ columns are swapped

This single detail is the entire measurement. Without it $T$ factorises as
$T_\text{cavity} \otimes T_\text{qubit}$, the qubit never becomes correlated with the
photon, and the readout record contains **no information about $n$ whatsoever** — the
HMM then just watches the cavity decay no matter what you feed it.

From the dispersive Hamiltonian (Eqn. 1),

$$H/\hbar = \omega_c a^\dagger a + \tfrac{1}{2}\left(\omega_q + 2\chi a^\dagger a\right)\sigma_z$$

the qubit frequency is shifted by $2\chi$ per photon. The parity measurement is a Ramsey
sequence that converts that frequency shift into a population:

| step | $n = 0$ | $n = 1$ |
|---|---|---|
| $\pi/2$ pulse | $\tfrac{1}{\sqrt2}(\lvert g\rangle + \lvert e\rangle)$ | $\tfrac{1}{\sqrt2}(\lvert g\rangle + \lvert e\rangle)$ |
| wait $t_p = \pi/\lvert 2\chi\rvert$ | nothing — no detuning | $\lvert e\rangle$ picks up $e^{-i\pi} = -1$ |
| $-\pi/2$ pulse | back to $\lvert g\rangle$ | ends in $\lvert e\rangle$ |

So the net operation is a **$\pi$ rotation of the qubit, conditioned on the photon** —
a bit flip, i.e. a CNOT with the cavity as control:

| qubit before | $n=0$ | $n=1$ |
|---|---|---|
| $\lvert g\rangle$ | $\lvert g\rangle$ | $\lvert e\rangle$ |
| $\lvert e\rangle$ | $\lvert e\rangle$ | $\lvert g\rangle$ |

Note it *flips* the qubit, it does not *set* it to $|e\rangle$. Nothing re-initialises
the qubit between cycles, so a photon that stays in the cavity **toggles the qubit
every cycle**. The signature of a photon is therefore an *alternating* record
$e, g, e, g, \ldots$, not a sustained string of $e$. An empty cavity leaves the qubit
alone and gives a constant string of $g$.

That is exactly what the swap encodes. Reading the $|1g\rangle$ row of Eqn. (2):

* $\to |1e\rangle = P_{11}P_{gg}$ — the cavity keeps the photon, the qubit does *not*
  err, and the parity map then flips it $g \to e$. This is the dominant term, $\approx 0.93$.
* $\to |1g\rangle = P_{11}P_{ge}$ — the cavity keeps the photon, the qubit *does* err,
  and the parity map flips it back. Staying in $|1g\rangle$ **requires an error**, which
  is why it carries the small probability.

In the photon columns, the "no error" outcome is the *flipped* qubit. In the $n=0$
columns there is no map to apply and everything is direct.

**Rule of thumb:** write the plain product (cavity) $\times$ (qubit), then swap the last
two columns.

*A note on the sign.* The paper's second pulse is $-\pi/2$. Using $+\pi/2$ instead is an
equally valid Ramsey interferometer, but it implements the *complementary* map — it flips
the qubit when $n = 0$ — and then the swap would belong in the $|0\cdot\rangle$ columns.
The choice is not cosmetic: qubit $T_1$ always drains toward $|g\rangle$, so under the
paper's convention a dead qubit fakes "no photon" (a harmless false negative), whereas
under $+\pi/2$ it fakes a photon with $P(n=1) = 1$. For a rare-event search that
asymmetry is decisive.

## 2. The three algorithms

All three are the same two ingredients — $T$ and $E$ — combined differently.

**Convention.** `T_row` is Eqn. (2) as printed, row-stochastic, used where probability
flows *backwards* along the arrows. `T_matrix = T_row.T` is column-stochastic, so that
`T_matrix @ p` propagates a probability *vector* forwards. Mixing them up does not raise
an error, it just silently gives the wrong answer.

### 2.1 Free evolution

No measurements at all — just the Markov chain, where we have a master equation for the probability vector $p_k$ of the hidden state at step $k$, and we know how to propagate it forward in time:

$$p_k = T^{\mathsf{T}} p_{k-1}$$

Useful as a sanity check (probability must stay normalised, the steady state must be
what you expect) and to see the bare timescales of the model.

### 2.2 Forward filter

Bayes' rule, applied once per measurement. Starting from your own formula,

$$P(X\mid Y) = \frac{P(Y\mid X)\,P(X)}{P(Y)}, \qquad
  P(Y) = \sum_i P(Y\mid X_i)\,P(X_i)$$

each step is,

$$p_k \;\propto\; \underbrace{D(R_k)}_{P(Y\mid X)}\;\underbrace{T^{\mathsf{T}} p_{k-1}}_{P(X)},
  \qquad p_k = \frac{D(R_k)\,T^{\mathsf{T}} p_{k-1}}{\big\lVert D(R_k)\,T^{\mathsf{T}} p_{k-1}\big\rVert_1}$$

In code this is literally one line. `T_matrix @ p` is the **prediction** step
(total probability, $P(X_k) = \sum_j P(X_k\mid X_j)P(X_j)$ — this is where the parity map
and the decoherence act); `D(R)` supplies $P(Y\mid X)$; and `np.sum(...)` is the
denominator $P(Y)$.

Note that `D(R) @ T_matrix @ p` is **not** a normalised probability vector. Its $i$-th
entry is the *joint* $P(X_i \cap Y)$, and summing over $i$ marginalises $X$ away to leave
$P(Y)$. That is why we divide.

The filter answers: **"given everything up to now, where am I now?"** It is causal — it
never looks at future measurements — so it is what you would run live at the fridge.

### 2.3 Backward algorithm — Eqn. (4)

The forward filter answers the wrong question for a dark matter search. We want to know
whether a photon was in the cavity **at the start**, when it was exposed, and by then we
already have the whole record in hand. Eqn. (4) of the paper is

$$
P(n_0) = \sum_{s_0 \in [|n_0 g\rangle, |n_0 e\rangle]} \sum_{s_1} \cdots \sum_{s_N}
   E_{s_0,R_0}\, T_{s_0,s_1}\, E_{s_1,R_1} \cdots T_{s_{N-1},s_N}\, E_{s_N,R_N}
$$

#### Building the recursion from scratch

Start from the easiest case and add one measurement at a time. Write
$E_{i,R} = P(Y{=}R \mid X{=}i)$ and $T_{ij} = P(X_{k+1}{=}j \mid X_k{=}i)$, and let

$$L_0(i) = P(R_0, R_1, \ldots, R_N \mid X_0 = i)$$

**One measurement.** Nothing to sum over yet:

$$L_0(i) = P(R_0 \mid X_0 = i) = E_{i,R_0}$$

**Two measurements.** Given $X_0 = i$, the readout $R_0$ depends only on $X_0$. Then the
chain hops to $X_1$, which emits $R_1$ — but $X_1$ is unknown, so sum over it:

$$L_0(i) = \underbrace{E_{i,R_0}}_{P(R_0\mid X_0=i)} \cdot \sum_j \underbrace{T_{ij}}_{P(X_1=j\mid X_0=i)} \underbrace{E_{j,R_1}}_{P(R_1\mid X_1=j)}$$

**Three measurements.** The same move again, nested one level deeper:

$$L_0(i) = E_{i,R_0} \sum_j T_{ij}\, E_{j,R_1} \sum_k T_{jk}\, E_{k,R_2}$$

**The compact form.** Those nested sums *are* a matrix product. With $D(R)$ the diagonal
matrix holding column $R$ of $E$, and $\mathbf{1}$ a vector of ones:

$$L_0 = D(R_0)\, T\, D(R_1)\, T\, D(R_2) \cdots T\, D(R_N)\, \mathbf{1}$$

That is Eqn. (4) written as matrices, and the "backward recursion" is nothing more than
**evaluating this product from the right**. The vector of ones is where you start because
to the right of $R_N$ there is nothing left to ask about.

#### What each step does

Reading right to left, every iteration does *two* things at once:

| operation | effect |
|---|---|
| $D(R_k)\,\beta$ | **absorb one more measurement** — multiply in the likelihood of the readout at step $k$ |
| $T\,(\cdots)$ | **sum over the unknown state and step the index back** — $\beta$ goes from being indexed by the state at step $k$ to the state at step $k-1$ |

The second one is the part that is invisible in the code: multiplying by $T$ *is* the
$\sum_j$ of the derivation, and it simultaneously moves the "given that" one step earlier.

#### Why $T$ and not $T^{\mathsf{T}}$

Because $T_{ij} = P(j \mid i)$, and here we want $\sum_j T_{ij}\, v(j)$ with the result
indexed by $i$ — which is exactly `T @ v`.

The forward filter wants the opposite, $\sum_i T_{ij}\, p(i)$ indexed by $j$, which is
`(T.T @ p)`. Same matrix, same chain of products; the only difference is which end you
start multiplying from. That is also why both give the same total evidence $P(R)$.

#### Back to the notation used in the code

The nested sums collapse into a backward recursion for
$\beta_k(i) = P(R_{k}, \ldots, R_N \mid X_{k-1} = i)$:

$$\beta_{N+1} = \mathbf{1}, \qquad \beta_k = T\,\big(D(R_k)\,\beta_{k+1}\big)$$

**$\beta$ is a likelihood, not a probability.** $\beta_0(i)$ is
$P(\text{this whole record} \mid \text{I started in state } i)$ — the reverse of what we
want, and its entries do not form a distribution over states. Flipping it is one more
application of the very same Bayes formula, now with the *entire record* playing the role
of $Y$ and $\beta$ playing the role of $P(Y\mid X)$:

$$P(X_0 = i \mid R) = \frac{\beta_0(i)\, p_0(i)}{\sum_j \beta_0(j)\, p_0(j)}$$

and then $P(n_0)$ is obtained by "summing over all possible initial qubit states", i.e.
adding the $|n_0 g\rangle$ and $|n_0 e\rangle$ components. Note Eqn. (4) opens with
$E_{s_0,R_0}$: the first readout is emitted by $s_0$ itself, with no transition before it.

Because each extra measurement multiplies the odds by a roughly constant likelihood
ratio, the log-odds grow **linearly** in $N$ and the probability of the wrong hypothesis
falls **exponentially**. This is a hypothesis test between two discrete alternatives, not
the estimation of a continuous parameter — hence $e^{-N}$ rather than $1/\sqrt{N}$. It is
what the paper means by "exponential rejection of false positives with only a linear cost
in measurement time".

### 2.4 Forward and backward do *not* converge to the same answer

This is the single easiest thing to get wrong, so it is worth being explicit.

|  | which state? | which measurements? |
|---|---|---|
| forward | $X_k$ — the **latest** one | $R_0 \ldots R_k$ — those seen so far |
| backward + Bayes flip | $X_0$ — the **first** one | $R_0 \ldots R_N$ — **all** of them |

Both are posteriors of the form $P(\text{state} \mid \text{measurements})$. What differs is
*which* state is being asked about. The forward filter at step $k = 29$ has seen all thirty
measurements, but it reports $P(X_{29} \mid R_0 \ldots R_{29})$ — the state *now* — not
$P(X_0 \mid \ldots)$.

Only the *recursion* runs in reverse, not the conditional. The one genuinely reversed
conditional is $\beta$ itself, $P(\text{record} \mid X_0)$, and that is an intermediate
quantity: the Bayes flip in `reconstruct_initial_state` turns it back into an ordinary
posterior.

**Why the forward saturates and the backward does not.** The forward is chasing a target
that keeps being re-randomised: at every step $T$ injects fresh uncertainty (the photon may
decay, the qubit may flip) at the same time as $D$ delivers fresh evidence. It settles at an
equilibrium where the information gained per measurement equals the uncertainty injected per
measurement — a plateau, here at $P(\text{wrong}) \approx 4\times 10^{-3}$.

The backward asks about a state that is frozen in the past. $X_0$ has already happened and
nothing can re-randomise it, so evidence simply accumulates and the error falls
exponentially without bound.

Put physically: to know whether a photon is in the cavity *right now*, old measurements are
useless, because the photon may have escaped after them. To know whether one was there *at
the start*, every alternation in the record counts, because the hypothesis "no photon at
step 0" has to explain all of them as readout errors.

The cell below demonstrates this directly — the forward literally forgets.


In [ ]:
def free_evolution(p0, n, T_matrix):
    """Evolve the state distribution for `n` cycles with NO measurements.

    p_k = T_matrix @ p_{k-1}, with T_matrix column-stochastic. Returns an
    (n, 4, 1) array; probas[0] is p0 itself, so index k is "after k cycles".
    """
    probas = np.empty((n, 4, 1))
    probas[0] = p0

    for i in range(1, n):
        probas[i] = T_matrix @ probas[i-1]

    return probas


def HMM_step(p, R, T_matrix, args):
    """One Bayes update: P(X_k | R_0..R_k) from P(X_{k-1} | R_0..R_{k-1}).

        T_matrix @ p  is the PREDICTION,  P(X) = sum_j P(X|X_j) P(X_j)
        D(R, args)    is the LIKELIHOOD,  P(Y|X)
        np.sum(x)     is the EVIDENCE,    P(Y) = sum_i P(Y|X_i) P(X_i)

    so this is literally P(X|Y) = P(Y|X) P(X) / P(Y).

    Note x itself is NOT a normalised probability vector: entry i is the joint
    P(X_i and Y). Summing over i marginalises X away and leaves P(Y), which is
    why we divide by it.
    """
    D_matrix = D(R, args)
    x = D_matrix @ T_matrix @ p

    return x / np.sum(x)


def HMM_simulation_normalized(p0, R_sequence, T_matrix, args):
    """Forward filter: returns P(X_k | R_0..R_k) for every k.

    Answers "is a photon here NOW?". Causal -- it never looks at future
    measurements -- so it is what you would run live at the fridge. Its downside
    is that it can always be fooled by the most recent readout error, so its
    error probability plateaus instead of falling exponentially. Use the
    backward algorithm when the question is about the initial state.

    The parity map lives inside T, so T is applied *before* every measurement,
    including the first one -- p0 is the state as prepared (qubit in |g>),
    before any parity mapping has happened.
    """
    n = len(R_sequence)
    probas = np.empty((n, 4, 1))

    p = p0
    for i in range(n):
        p = HMM_step(p, R_sequence[i], T_matrix, args)
        probas[i] = p

    return probas


def backward_step(beta, R, T_row, args):
    """One step of the backward recursion:  beta_k = T (D(R_k) beta_{k+1}).

    CAREFUL with the orientation of T. This function needs the ROW-stochastic T
    of Eqn. (2), because likelihood flows backwards along the arrows. It is
    called below with T_matrix (= T_row.T), and the `.T` here undoes that -- so
    the two transposes cancel and the recursion is correct.

    The division by np.sum(x) is only a rescaling to stop beta underflowing; it
    cancels when the posterior is normalised. It does NOT make beta a
    probability distribution -- beta is a likelihood.
    """
    D_matrix = D(R, args)
    x = T_row.T @ D_matrix @ beta

    return x / np.sum(x)


def backward_algorithm(R_sequence, T_matrix, args):
    """Backward algorithm: the nested sums of Eqn. (4), accumulated from the end.

    Returns an (n, 4, 1) array of betas, where

        beta_k(i) = P(R_k ... R_{N} | X_{k-1} = i)

    i.e. "how likely was this stretch of record, SUPPOSING I was in state i".
    Note the storage order is reversed: betas[j] is the beta after folding in
    j+1 measurements counted from the END, so betas[-1] is the fully
    accumulated beta_0 that uses the whole record.

    beta is a LIKELIHOOD, not a probability of the state -- do not plot its
    components as if they were P(X). Use reconstruct_initial_state() to turn it
    into a posterior.
    """
    n = len(R_sequence)
    betas = np.empty((n, 4, 1))

    beta = np.ones((4, 1))
    for i in range(n-1, -1, -1):
        beta = backward_step(beta, R_sequence[i], T_matrix, args)
        betas[n-1-i] = beta

    return betas


def backward_algorithm_recursive(R_sequence, T_matrix, args):
    
    def rec_step(i):
        if i == len(R_sequence):
            return np.ones((4, 1))
        else:
            beta_next = rec_step(i + 1)
            return backward_step(beta_next, R_sequence[i], T_matrix, args)
    
    return np.array([rec_step(i) for i in range(len(R_sequence)-1, -1, -1)])

def reconstruct_initial_state(p0, R_sequence, T_matrix, args):
    """Eqn. (4): P(X_0 | R_0 ... R_N) -- the quantity Fig. 2(c) plots.

        P(n_0) = sum_{s_0 in [|n_0 g>, |n_0 e>]} sum_{s_1}...sum_{s_N}
                     E_{s_0,R_0} T_{s_0,s_1} E_{s_1,R_1} ... T_{s_{N-1},s_N} E_{s_N,R_N}

    backward_algorithm() computes the inner sums. What it returns is
    P(record | started in state i) -- the REVERSE of what we want. Flipping it
    is one more application of Bayes, with the whole record playing the role of
    Y and beta playing the role of P(Y|X):

        P(X_0 = i | R) = beta_0(i) p0(i) / sum_j beta_0(j) p0(j)

    Two factors are still owed to beta and are supplied here:

      * the leading E_{s_0,R_0}  -- Eqn. (4) opens with it, because the first
        readout is emitted by s_0 itself with no transition before it
      * the prior p0 over s_0

    Then "summing over all possible initial qubit states" is post[0]+post[1]
    for n_0 = 0, and post[2]+post[3] for n_0 = 1.
    """
    beta = backward_algorithm(R_sequence, T_matrix, args)[-1].reshape(4, 1)
    post = p0 * (D(R_sequence[0], args) @ beta)

    return post / np.sum(post)


def likelihood_ratio(Pn0, Pn1):
    """Decision statistic: how many times more likely is "photon" than "empty".

    Fed with the posteriors from reconstruct_initial_state() under a flat 50/50
    prior, the posterior odds equal the likelihood ratio P(R|n0=1)/P(R|n0=0).
    """
    return Pn1 / Pn0

---

## 3. Free evolution — no measurements

First a deliberately simplified parameter set: $T_2^q = \infty$ and $\bar n_q = 0$, so the
only error channel left is qubit relaxation. This makes the dynamics easy to read before
turning everything on.

In [ ]:
# Simplified parameters: no dephasing and no thermal qubit population, so the
# only error channel left is qubit T1 decay. Device values are from the
# Supplemental Material of the paper.
#Qubit constants
T1_q = 108e-6
# T2_q = 61e-6
T2_q = np.inf
# n_q = 5.1e-2
n_q = 0.0

#Cavity constants
T1_c = 546e-6
n_c = 0.0

#Measurement constants
t_m = 10e-6
t_p = 380e-9

# Readout fidelities
Fgg = 0.958
Fee = 0.953

args = {
    "T1_q": T1_q,
    "T2_q": T2_q,
    "n_q": n_q,
    "T1_c": T1_c,
    "n_c": n_c,
    "Fgg": Fgg,
    "Fee": Fee,
    "Fge": 1 - Fgg,
    "Feg": 1 - Fee
}

In [ ]:
# |0g>, |0e>, |1g>, |1e>
p0 = np.array([[0, 0, 0, 1]]).T

T_row    = T(t_m, t_p, args)      # row-stochastic, for the backward pass
T_matrix = T_row.T                # column-stochastic, for T @ p

The eigenvalues of $T^{\mathsf{T}}$ tell you the whole story before you plot anything:

* $\lambda = 1$ — the steady state (probability is conserved, so this always exists).
* $\lambda = e^{-t_m/T_1^s} \approx 0.982$ — the cavity emptying.
* A **negative** eigenvalue $\lambda \approx -0.895$ — this is the parity map toggling the
  qubit. $(-0.895)^k$ changes sign at every step, so it is a period-2 oscillation, and its
  magnitude $\lvert\lambda\rvert = (1 - P_{ge} - P_{eg})\cdot e^{-t_m/T_1^s}$ says the
  oscillation dies when *either* the photon leaves *or* the qubit errs.

In [ ]:
eigenvalues, eigenvectors = np.linalg.eig(T_matrix)

print("Eigenvalues:", eigenvalues)

In [ ]:
steady_state_vector = eigenvectors[:, np.isclose(eigenvalues, 1)]
steady_state_vector /= np.sum(steady_state_vector)  # Normalize the steady state vector

print("Steady state vector: \n", steady_state_vector)

In [ ]:
n_times = 250

probas = free_evolution(p0, n_times, T_matrix)

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(12, 8))
state_labels = ['|0g>', '|0e>', '|1g>', '|1e>']
for i in range(4):
    ax[i//2, i%2].plot(probas[:, i], label=state_labels[i])
    ax[i//2, i%2].set_title(f'Probability of {state_labels[i]}')
    ax[i//2, i%2].set_xlabel('Time step')
    ax[i//2, i%2].set_ylabel('Probability')
    ax[i//2, i%2].legend()
    ax[i//2, i%2].grid()
plt.tight_layout()

print("Final probabilities after n_times steps:", probas[-1])
print('Suma de probabilidades:', np.sum(probas[-1]))

**Reading the plot.** We started in $|1e\rangle$, i.e. with a photon present. While the
photon is there, the parity map flips the qubit every single cycle, so $|1g\rangle$ and
$|1e\rangle$ oscillate against each other with period 2 — this is the same
$e, g, e, g, \ldots$ alternation as the readout record, seen as populations rather than as
a record. The envelope decays with the negative eigenvalue above.

The $|0g\rangle$ and $|0e\rangle$ panels are smooth precisely *because* there is no photon
there: no photon, no conditional flip, nothing to oscillate. $|0e\rangle$ is a transient —
population arriving from $|1e\rangle$ when the photon decays while the qubit happens to be
excited — which then relaxes into $|0g\rangle$. With $\bar n_q = 0$ nothing repopulates it,
so it ends at zero.

The oscillation is not a numerical artefact. It is the signal.

### 3.2 Full qubit parameters

Now with the real numbers: $T_2^q = 61\,\mu s$ and $\bar n_q = 5.1\times10^{-2}$. The
steady state is no longer pure $|0g\rangle$, because the thermal qubit population keeps
$|0e\rangle$ alive at $\approx \bar n_q$.

In [ ]:
# The real device parameters (Dixit et al., Supplemental Material).
#Qubit constants
T1_q = 108e-6
T2_q = 61e-6
n_q = 5.1e-2

#Cavity constants
T1_c = 546e-6
n_c = 0.0

#Measurement constants
t_m = 10e-6
t_p = 380e-9

# Readout fidelities
Fgg = 0.958
Fee = 0.953

args = {
    "T1_q": T1_q,
    "T2_q": T2_q,
    "n_q": n_q,
    "T1_c": T1_c,
    "n_c": n_c,
    "Fgg": Fgg,
    "Fee": Fee,
    "Fge": 1 - Fgg,
    "Feg": 1 - Fee
}

In [ ]:
# |0g>, |0e>, |1g>, |1e>
p0 = np.array([[0, 0, 0, 1]]).T

T_row    = T(t_m, t_p, args)      # row-stochastic, for the backward pass
T_matrix = T_row.T                # column-stochastic, for T @ p

In [ ]:
probas = free_evolution(p0, n_times, T_matrix)

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(12, 8))
state_labels = ['|0g>', '|0e>', '|1g>', '|1e>']
for i in range(4):
    ax[i//2, i%2].plot(probas[:, i], label=state_labels[i])
    ax[i//2, i%2].set_title(f'Probability of {state_labels[i]}')
    ax[i//2, i%2].set_xlabel('Time step')
    ax[i//2, i%2].set_ylabel('Probability')
    ax[i//2, i%2].legend()
    ax[i//2, i%2].grid()
plt.tight_layout()

print("Final probabilities after n_times steps:", probas[-1])
print('Suma de probabilidades:', np.sum(probas[-1]))

---

## 4. Forward filter

Now we condition on a record. The two idealised records are, from §1.4:

* `R_alternating` = $e,g,e,g,\ldots$ — what one photon in the cavity produces.
* `R_constant_g` = $g,g,g,\ldots$ — what an empty cavity produces.

Each is fed to the filter together with the matching preparation, and we marginalise the
qubit away to get $P(n_k = 0)$ and $P(n_k = 1)$.

In [ ]:
# Same device parameters, restated so this section can be run on its own.
#Qubit constants
T1_q = 108e-6
T2_q = 61e-6
n_q = 5.1e-2

#Cavity constants
T1_c = 546e-6
n_c = 0.0

#Measurement constants
t_m = 10e-6
t_p = 380e-9

# Readout fidelities
Fgg = 0.958
Fee = 0.953

args = {
    "T1_q": T1_q,
    "T2_q": T2_q,
    "n_q": n_q,
    "T1_c": T1_c,
    "n_c": n_c,
    "Fgg": Fgg,
    "Fee": Fee,
    "Fge": 1 - Fgg,
    "Feg": 1 - Fee
}

In [ ]:
T_row    = T(t_m, t_p, args)   # row-stochastic, Eqn. (2) as printed
T_matrix = T_row.T             # column-stochastic, for T_matrix @ p


N = 30
R_alternating = np.array([(k + 1) % 2 for k in range(N)])   # 1,0,1,0,...
R_constant_g  = np.zeros(N, dtype=int)                      # 0,0,0,...

p0_0 = np.array([[1-n_q, n_q, 0, 0]]).T   # cavity prepared empty
p0_1 = np.array([[0, 0, 1-n_q, n_q]]).T   # cavity prepared with one photon

probas_0 = HMM_simulation_normalized(p0_0, R_constant_g,  T_matrix, args)
probas_1 = HMM_simulation_normalized(p0_1, R_alternating, T_matrix, args)

p0_n0 = probas_0[:, 0] + probas_0[:, 1]
p0_n1 = probas_0[:, 2] + probas_0[:, 3]
p1_n0 = probas_1[:, 0] + probas_1[:, 1]
p1_n1 = probas_1[:, 2] + probas_1[:, 3]

print("empty  cavity, constant-g  record -> P(n=1):", p0_n1.ravel()[[0, 4, 9, 29]])
print("photon cavity, alternating record -> P(n=1):", p1_n1.ravel()[[0, 4, 9, 29]])

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(12, 8))
state_labels = ['|0g>', '|0e>', '|1g>', '|1e>']
for i in range(4):
    ax[i//2, i%2].plot(probas_0[:, i], label=state_labels[i])
    ax[i//2, i%2].set_title(f'Probability of {state_labels[i]}')
    ax[i//2, i%2].set_xlabel('Time step')
    ax[i//2, i%2].set_ylabel('Probability')
    ax[i//2, i%2].legend()
    ax[i//2, i%2].grid()
    
fig.suptitle("Cavity prepared empty")
plt.tight_layout()

print("Final probabilities after n_times steps:", probas_0[-1])
print('Suma de probabilidades:', np.sum(probas_0[-1]))

print("Final probabilities of measuring 0 photons:", p0_n0[-1])
print("Final probabilities of measuring 1 photons:", p0_n1[-1])

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(12, 8))
state_labels = ['|0g>', '|0e>', '|1g>', '|1e>']
for i in range(4):
    ax[i//2, i%2].plot(probas_1[:, i], label=state_labels[i])
    ax[i//2, i%2].set_title(f'Probability of {state_labels[i]}')
    ax[i//2, i%2].set_xlabel('Time step')
    ax[i//2, i%2].set_ylabel('Probability')
    ax[i//2, i%2].legend()
    ax[i//2, i%2].grid()
    
fig.suptitle("Cavity prepared with one photon")
plt.tight_layout()

print("Final probabilities after n_times steps:", probas_1[-1])
print('Suma de probabilidades:', np.sum(probas_1[-1]))

print("Final probabilities of measuring 0 photons:", p1_n0[-1])
print("Final probabilities of measuring 1 photons:", p1_n1[-1])

---

## 5. Backward algorithm — reconstructing the initial state

This is Fig. 2(c) of the paper: the reconstructed **initial** cavity state as a function of
how many measurements went into the reconstruction.

`backward_algorithm` returns the raw $\beta$ likelihoods. On their own they are *not*
probabilities of the state and must not be plotted as such — they answer
"how likely was this record *supposing* I started in state $i$", which is the reverse
conditional. `reconstruct_initial_state` performs the final Bayes flip (multiply by the
leading $E_{s_0,R_0}$ and by the prior, then normalise) and is what we plot.

We use an agnostic 50/50 prior over the cavity, so the record does all the work rather than
the prior assuming the answer.

In [ ]:
# Reconstruct the INITIAL cavity state from the first k measurements, k = 1..N.
# An agnostic 50/50 prior over the cavity, so the record does the work and the
# prior does not assume the answer.
prior = 0.5 * p0_0 + 0.5 * p0_1
ks = np.arange(1, N + 1)

post_0 = np.array([reconstruct_initial_state(prior, R_constant_g[:k],  T_matrix, args).ravel()
                   for k in ks])
post_1 = np.array([reconstruct_initial_state(prior, R_alternating[:k], T_matrix, args).ravel()
                   for k in ks])

# marginalise the initial qubit state away -- the "sum over initial qubit
# states" that Eqn. (4) calls for
p0_n0_backward = post_0[:, 0] + post_0[:, 1]
p0_n1_backward = post_0[:, 2] + post_0[:, 3]
p1_n0_backward = post_1[:, 0] + post_1[:, 1]
p1_n1_backward = post_1[:, 2] + post_1[:, 3]

print("empty  cavity -> P(n0=1) after 1, 5, 10, 30 measurements:",
      p0_n1_backward[[0, 4, 9, 29]])
print("photon cavity -> P(n0=0) after 1, 5, 10, 30 measurements:",
      p1_n0_backward[[0, 4, 9, 29]])

In [ ]:
# Fig. 2(c): reconstructed initial cavity state vs how many measurements were used
fig, ax = plt.subplots(1, 2, figsize=(13, 5), sharey=True)

for a, (Pn0, Pn1, lbl) in zip(
        ax,
        [(p0_n0_backward, p0_n1_backward, r"record $g,g,g,\dots$   (cavity was empty)"),
         (p1_n0_backward, p1_n1_backward, r"record $e,g,e,g,\dots$   (cavity had a photon)")]):
    a.semilogy(ks, Pn0, "o-", ms=3, color="blue",   label=r"$P(n_0=0)$")
    a.semilogy(ks, Pn1, "o-", ms=3, color="orange", label=r"$P(n_0=1)$")
    a.set_title(lbl, fontsize=10)
    a.set_xlabel("number of measurements used")
    a.grid(alpha=.3, which="both")
    a.legend()

ax[0].set_ylabel("reconstructed initial cavity state")
ax[0].set_ylim(1e-20, 3)
plt.tight_layout()

**Reading the plot.** The wrong hypothesis is suppressed exponentially — a straight line on
a log axis — which is the central claim of the technique.

The two panels are asymmetric, and the asymmetry is physical:

* Right panel (a photon was there): $P(n_0=0)$ falls by ~18 orders of magnitude over 30
  measurements. To fake an alternating record with an empty cavity you would need a readout
  error on every *other* measurement, $\approx 15$ of them, each of probability
  $F_{g\mathcal{E}} = 0.042$; and $0.042^{15} \approx 2\times10^{-21}$. The errors have to
  conspire, and that is where the exponential comes from.

* Left panel (the cavity was empty): $P(n_0=1)$ drops but **floors at**
  $\approx 1.9\times10^{-2}$. This is not a bug and no amount of extra measurement removes
  it: a photon that decays during the very first $t_m$ produces a record of pure $g$,
  identical to an empty cavity. The floor is exactly $P_{10} = 1 - e^{-t_m/T_1^s} = 0.018$,
  and it only improves with a better cavity or a shorter $t_m$. This is the sense in which
  the paper's detector is "limited by a residual background of real photons".

### 5.1 The forward filter forgets — a demonstration

In [ ]:
# Demonstration: the forward filter forgets, the backward one does not.
flat = np.array([[0.25, 0.25, 0.25, 0.25]]).T

full = HMM_simulation_normalized(prior, R_alternating, T_matrix, args)[-1].ravel()
print("FORWARD at step 29, using ALL 30 measurements")
print(f"   P(n=1) = {full[2]+full[3]:.10f}   P(wrong) = {full[0]+full[1]:.3e}\n")

for start in (20, 24, 27):
    # restart from a flat prior partway through: the early record is DISCARDED
    part = HMM_simulation_normalized(flat, R_alternating[start:], T_matrix, args)[-1].ravel()
    print(f"FORWARD at step 29, discarding the first {start} measurements "
          f"(only {30-start} used)")
    print(f"   P(n=1) = {part[2]+part[3]:.10f}   "
          f"difference from using all 30 = {abs((part[2]+part[3])-(full[2]+full[3])):.3e}")

print("\n-> Discarding two thirds of the data moves the forward answer by ~1e-6.")
print("   The chain mixes, so old evidence says nothing about where you are NOW,")
print("   and the filter plateaus.\n")

print("BACKWARD, P(n_0 = 1), the same measurements added one at a time:")
for k in (5, 10, 20, 30):
    b = reconstruct_initial_state(prior, R_alternating[:k], T_matrix, args).ravel()
    print(f"   first {k:2d} measurements -> P(wrong) = {b[0]+b[1]:.3e}")
print("\n-> Same data, 18 orders of magnitude, no plateau: X_0 is frozen in the past")
print("   and cannot be re-randomised, so evidence just accumulates.")

---

## 6. Likelihood ratio and detection threshold

The decision statistic for a search is the ratio

$$\Lambda = \frac{P(n_0 = 1 \mid R)}{P(n_0 = 0 \mid R)}$$

With the 50/50 prior used above this posterior odds ratio equals the likelihood ratio
$P(R \mid n_0=1) / P(R \mid n_0=0)$, so the threshold can be read directly as "how many
times more likely is this record if a photon was there".

A candidate is accepted when $\Lambda$ exceeds a threshold. Raising the threshold trades
efficiency (you throw away real photons) against false positives — and because $\Lambda$
grows exponentially in the number of measurements, buying orders of magnitude of rejection
costs only linearly more time.

In [ ]:
likelihood_ratios_0 = likelihood_ratio(p0_n0_backward, p0_n1_backward)
likelihood_ratios_1 = likelihood_ratio(p1_n0_backward, p1_n1_backward)

likelihood_treshold = 1e5   # accept a photon candidate above this

print("empty  cavity: LR after 30 measurements =", f"{likelihood_ratios_0[-1]:.3e}",
      "-> accepted" if likelihood_ratios_0[-1] > likelihood_treshold else "-> rejected")
print("photon cavity: LR after 30 measurements =", f"{likelihood_ratios_1[-1]:.3e}",
      "-> accepted" if likelihood_ratios_1[-1] > likelihood_treshold else "-> rejected")

n_needed = np.argmax(likelihood_ratios_1 > likelihood_treshold) + 1
print(f"measurements needed to cross the threshold with a photon present: {n_needed}")

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

ax[0].plot(ks, likelihood_ratios_0, label='Likelihood ratio, cavity empty', color='blue')
ax[0].axhline(y=likelihood_treshold, color='red', linestyle='--', label='Threshold')
ax[0].set_title('Empty cavity -- the statistic must stay BELOW threshold')
ax[0].set_ylabel(r'$\Lambda = P(n_0{=}1)\,/\,P(n_0{=}0)$')
ax[0].set_yscale('log')
ax[0].legend(); ax[0].grid(alpha=.3)

ax[1].plot(ks, likelihood_ratios_1, label='Likelihood ratio, one photon', color='orange')
ax[1].axhline(y=likelihood_treshold, color='red', linestyle='--', label='Threshold')
ax[1].set_title('One photon -- the statistic grows exponentially, crossing the threshold')
ax[1].set_xlabel('number of measurements used')
ax[1].set_ylabel(r'$\Lambda = P(n_0{=}1)\,/\,P(n_0{=}0)$')
ax[1].set_yscale('log')
ax[1].legend(); ax[1].grid(alpha=.3)

plt.tight_layout()

---

## 7. What these numbers do and do not mean

A posterior of $1 - 10^{-18}$ is **not** a measurement precision. It is a ratio of
likelihoods between two hypotheses, computed *inside* the model, and it is exactly as
trustworthy as the model's tails.

The exponential rests entirely on the likelihood ratios **multiplying**, i.e. on the
measurement errors being independent from one cycle to the next. Real devices violate this:
quasiparticle bursts, ionising radiation, TLS switching and heating events flip whole *runs*
of measurements at once. Such an event contributes a single likelihood ratio, not fifteen,
and its rate is nowhere near $10^{-21}$. The true false-positive floor is set by the rate of
those correlated events, which no amount of modelling can give you — it has to be measured.

The paper is careful about this. It does not claim $10^{-21}$; it claims **15.7 dB** below
the standard quantum limit, with the detector "limited by a residual background of real
photons". The achievement is that the HMM removes *detector* errors thoroughly enough that
what remains is real photons: the measurement becomes background-limited instead of
detector-limited, and that is what enables the quoted factor of 1300 speed-up.

**Practical consequence:** never quote the HMM's $10^{-18}$ as a false-positive rate. The
HMM provides the optimal *statistic*; the *distribution* of that statistic under the null
hypothesis must be calibrated empirically — run the detector on a known-empty cavity many
times and count how often the reconstruction crosses the threshold.

### Known discrepancy in the paper's parameters

The text states $\lvert 2\chi\rvert = 2\pi \times 1.13\,$MHz and $t_p = \pi/\lvert 2\chi\rvert$,
which gives $t_p = 442.5\,$ns, but the parameter table used for the HMM lists
$t_p = 380\,$ns. At $380\,$ns the accumulated phase would be $0.859\pi$ rather than $\pi$,
and the parity map would not be a perfect permutation ($P(g \to e \mid \text{photon}) = 0.95$
instead of 1).

This does not affect anything computed here: $t_p$ enters the model *only* through
$P_\phi = 1 - e^{-t_p/T_2^q}$. The rotation angle never appears — the model assumes the
permutation is exact and absorbs any loss of contrast into the readout fidelities. Using
$380\,$ns as the dephasing window, as done above, is correct.